# Phase 1: Data and baselines

Load BTC-USD daily data, time-based train/val/test split, two baselines (last value, moving average), and report MAE, RMSE, directional accuracy.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path
import sys

ROOT = Path('.').resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
elif ROOT.name != 'crypto-price-prediction':
    ROOT = ROOT / 'crypto-price-prediction'
sys.path.insert(0, str(ROOT))
from src.metrics import regression_metrics

## 1. Download and clean data

In [ ]:
TICKER = 'BTC-USD'
START = '2020-01-01'
END = None  # up to now
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)
cache_path = DATA_DIR / f'{TICKER.replace("-", "_")}_daily.parquet'

if cache_path.exists():
    df = pd.read_parquet(cache_path)
    print('Loaded from cache:', cache_path)
else:
    raw = yf.download(TICKER, start=START, end=END, progress=False, auto_adjust=True)
    if raw.index.nlevels > 1:
        raw = raw.reset_index(level=1, drop=True)
    raw.index = pd.to_datetime(raw.index).tz_localize(None)
    raw = raw.sort_index()
    raw = raw.ffill().dropna()
    df = raw[['Close']].copy()
    df.columns = ['price']
    if 'Volume' in raw.columns:
        df['volume'] = raw['Volume']
    df.to_parquet(cache_path)
    print('Downloaded and saved to', cache_path)
print(df.head(), df.shape)

## 2. Time-based split (70 / 15 / 15)

In [ ]:
n = len(df)
train_end = int(0.70 * n)
val_end = int(0.85 * n)

train = df.iloc[:train_end]
val = df.iloc[train_end:val_end]
test = df.iloc[val_end:]

print('Train', train.index[0], '->', train.index[-1], len(train))
print('Val  ', val.index[0], '->', val.index[-1], len(val))
print('Test ', test.index[0], '->', test.index[-1], len(test))

## 3. Next-day target and baselines on test set

In [ ]:
y_true = test['price'].values[1:]
y_prev = test['price'].values[:-1]

# Baseline 1: last value (predict tomorrow = today)
pred_last = y_prev

# Baseline 2: 7-day moving average (MA of past 7 days predicts next day)
prices = test['price'].values
window = 7
pred_ma = np.array([np.mean(prices[i - window:i]) for i in range(window, len(prices))])
y_true_ma = y_true[window - 1:]  # same length as pred_ma

m_last = regression_metrics(y_true, pred_last)
m_ma = regression_metrics(y_true_ma, pred_ma)

print('Baseline 1 (last value):', m_last)
print('Baseline 2 (7-day MA): ', m_ma)

In [ ]:
print('Test set metrics')
print('                 MAE       RMSE   Dir.Acc')
print('Last value  ', f"{m_last['mae']:>10.2f}", f"{m_last['rmse']:>10.2f}", f"{m_last['directional_accuracy']:>8.2%}")
print('7-day MA     ', f"{m_ma['mae']:>10.2f}", f"{m_ma['rmse']:>10.2f}", f"{m_ma['directional_accuracy']:>8.2%}")